In [1]:
import sys
from pathlib import Path
repo_root = Path().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# nb_img_02 — Unsupervised Clustering on CLIP Embeddings

Runs KMeans, Agglomerative, and DEC clustering on the 512-dim CLIP embeddings.
Outputs label CSVs, metrics JSONs, and raw cluster assignment .npy files.

In [2]:
# Section 1 — Load embeddings and manifest
import numpy as np
import pandas as pd
from utils.image_config import IMAGE_MANIFEST_PATH, IMAGE_EMBEDDINGS_PATH, IMAGE_CLASS_NAMES, IMAGE_NUM_CLASSES, IMAGE_SEED, RESULTS_DIR

manifest = pd.read_csv(IMAGE_MANIFEST_PATH)
embeddings = np.load(IMAGE_EMBEDDINGS_PATH)
print(f"Loaded {len(manifest)} rows, embeddings {embeddings.shape}")

Loaded 5500 rows, embeddings (5500, 512)


In [3]:
# Section 2 — UMAP reduction (shared for KMeans and Agglomerative)
import umap

reducer = umap.UMAP(n_components=30, metric="cosine", random_state=IMAGE_SEED)
embeddings_reduced = reducer.fit_transform(embeddings).astype(np.float32)
print(f"UMAP reduced: {embeddings_reduced.shape}")

C:\python3.11.8\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\python3.11.8\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP reduced: (5500, 30)


In [4]:
# Section 3 — KMeans clustering
from sklearn.cluster import KMeans
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score, f1_score
from sklearn.metrics import silhouette_score
import json

def hungarian_match(pred_labels, true_labels, n_classes):
    """Map cluster ids to class ids via Hungarian algorithm. Returns remapped pred_labels."""
    cost = np.zeros((n_classes, n_classes))
    for i in range(n_classes):
        for j in range(n_classes):
            cost[i, j] = -np.sum((pred_labels == i) & (true_labels == j))
    row_ind, col_ind = linear_sum_assignment(cost)
    mapping = {row_ind[k]: col_ind[k] for k in range(len(row_ind))}
    return np.array([mapping.get(p, 0) for p in pred_labels])

true_labels = manifest["label_idx"].to_numpy()

km = KMeans(n_clusters=IMAGE_NUM_CLASSES, n_init=10, random_state=IMAGE_SEED)
km_raw = km.fit_predict(embeddings_reduced)
km_labels = hungarian_match(km_raw, true_labels, IMAGE_NUM_CLASSES)

acc_km = (km_labels == true_labels).mean()
print(f"KMeans ACC: {acc_km:.4f}")

KMeans ACC: 0.9922


In [5]:
# Section 4 — Agglomerative clustering
from sklearn.cluster import AgglomerativeClustering

agg = AgglomerativeClustering(n_clusters=IMAGE_NUM_CLASSES, linkage="ward")
agg_raw = agg.fit_predict(embeddings_reduced)
agg_labels = hungarian_match(agg_raw, true_labels, IMAGE_NUM_CLASSES)

acc_agg = (agg_labels == true_labels).mean()
print(f"Agglomerative ACC: {acc_agg:.4f}")

Agglomerative ACC: 0.9922


In [ ]:
# Section 5 — DEC clustering
from utils.dec import train_dec

dec_raw, _, _ = train_dec(embeddings, n_clusters=IMAGE_NUM_CLASSES, seed=IMAGE_SEED)
dec_labels = hungarian_match(dec_raw, true_labels, IMAGE_NUM_CLASSES)

acc_dec = (dec_labels == true_labels).mean()
print(f"DEC ACC: {acc_dec:.4f}")

In [7]:
# Section 6 — Save all results
RESULTS_DIR.mkdir(exist_ok=True)
sil_sample = min(2000, len(embeddings_reduced))
sil_idx = np.random.default_rng(IMAGE_SEED).choice(len(embeddings_reduced), sil_sample, replace=False)

for method_name, pred_labels, emb_for_sil in [
    ("kmeans", km_labels, embeddings_reduced),
    ("agglomerative", agg_labels, embeddings_reduced),
    ("dec", dec_labels, embeddings),
]:
    # predictions CSV
    out_df = manifest[["filepath", "true_label"]].copy()
    out_df["predicted_label"] = [IMAGE_CLASS_NAMES[i] for i in pred_labels]
    out_df.to_csv(RESULTS_DIR / f"image_full_labels_unsup_{method_name}.csv", index=False)

    # cluster assignments .npy (raw cluster ids before Hungarian, needed by nb_img_03b)
    if method_name == "kmeans":
        np.save(RESULTS_DIR / f"image_cluster_assignments_kmeans.npy", km_raw)
    elif method_name == "agglomerative":
        np.save(RESULTS_DIR / f"image_cluster_assignments_agglomerative.npy", agg_raw)
    else:
        np.save(RESULTS_DIR / f"image_cluster_assignments_dec.npy", dec_raw)

    # metrics JSON
    sil = silhouette_score(emb_for_sil[sil_idx], pred_labels[sil_idx], metric="cosine")
    metrics = {
        "acc": float((pred_labels == true_labels).mean()),
        "f1": float(f1_score(true_labels, pred_labels, average="macro", zero_division=0)),
        "nmi": float(normalized_mutual_info_score(true_labels, pred_labels)),
        "ari": float(adjusted_rand_score(true_labels, pred_labels)),
        "silhouette": float(sil),
    }
    with open(RESULTS_DIR / f"image_metrics_unsup_{method_name}.json", "w") as f:
        json.dump(metrics, f, indent=2)
    print(f"{method_name}: ACC={metrics['acc']:.4f}  F1={metrics['f1']:.4f}  NMI={metrics['nmi']:.4f}")

kmeans: ACC=0.9922  F1=0.9922  NMI=0.9789
agglomerative: ACC=0.9922  F1=0.9922  NMI=0.9789


dec: ACC=0.9938  F1=0.9938  NMI=0.9832
